In [1]:
import nltk
import numpy as np
import pandas as pd
import string as st
import re

from sklearn.preprocessing import LabelEncoder
from nltk import PorterStemmer, WordNetLemmatizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer

In [2]:
data = pd.read_csv('dataset.csv', encoding='utf-8')
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 201 entries, 0 to 200
Data columns (total 6 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   query             201 non-null    object
 1   long_common_name  201 non-null    object
 2   component         201 non-null    object
 3   system            201 non-null    object
 4   property          201 non-null    object
 5   relevance         201 non-null    int64 
dtypes: int64(1), object(5)
memory usage: 9.6+ KB


In [3]:
encoders = {}
for col in ['query','component', 'system', 'property']:
  le = LabelEncoder()
  data[col+'_cat'] = le.fit_transform(data[col])
  encoders[col] = le
data.head()

,query,long_common_name,component,system,property,relevance,query_cat,component_cat,system_cat,property_cat
0,glucose in blood,C reactive protein [Mass/volume] in Serum or P...,C reactive protein,Ser/Plas,MCnc,0,1,20,8,3
1,glucose in blood,Bicarbonate [Moles/volume] in Blood,Bicarbonate,Bld,SCnc,0,1,11,0,11
2,glucose in blood,Rh [Type] in Blood,Rh,Bld,Type,0,1,51,0,14
3,glucose in blood,Trimethoprim+Sulfamethoxazole [Susceptibility],Trimethoprim+Sulfamethoxazole,Isolate,Susc,0,1,53,4,12
4,glucose in blood,Bilirubin.total [Mass/volume] in Serum or Plasma,Bilirubin,Ser/Plas,MCnc,0,1,12,8,3


In [4]:
def remove_punct(text):
    return ("".join([ch for ch in text if ch not in st.punctuation]))

def tokenize(text):
    text = re.split(r'\s+' ,text)
    return [x.lower() for x in text]

def remove_small_words(text):
    return [x for x in text if len(x) > 3 ]

def remove_stopwords(text):
    return [word for word in text if word not in nltk.corpus.stopwords.words('english')]

def lemmatize(text):
    word_net = WordNetLemmatizer()
    return [word_net.lemmatize(word) for word in text]

def return_sentences(tokens):
    return " ".join([word for word in tokens])

In [5]:
data['removed_punc'] = data['long_common_name'].apply(lambda x: remove_punct(x))
data['tokens'] = data['removed_punc'].apply(lambda msg : tokenize(msg))
#data['larger_tokens'] = data['tokens'].apply(lambda x : remove_small_words(x))
data['clean_tokens'] = data['tokens'].apply(lambda x : remove_stopwords(x))
data['lemma_words'] = data['clean_tokens'].apply(lambda x : lemmatize(x))
data['clean_text'] = data['lemma_words'].apply(lambda x : return_sentences(x))
data.head().transpose()

,0,1,2,3,4
query,glucose in blood,glucose in blood,glucose in blood,glucose in blood,glucose in blood
long_common_name,C reactive protein [Mass/volume] in Serum or P...,Bicarbonate [Moles/volume] in Blood,Rh [Type] in Blood,Trimethoprim+Sulfamethoxazole [Susceptibility],Bilirubin.total [Mass/volume] in Serum or Plasma
component,C reactive protein,Bicarbonate,Rh,Trimethoprim+Sulfamethoxazole,Bilirubin
system,Ser/Plas,Bld,Bld,Isolate,Ser/Plas
property,MCnc,SCnc,Type,Susc,MCnc
relevance,0,0,0,0,0
query_cat,1,1,1,1,1
component_cat,20,11,51,53,12
system_cat,8,0,0,4,8
property_cat,3,11,14,12,3


In [6]:
tfidf = TfidfVectorizer(max_features=20)
data['FV_document'] = tfidf.fit_transform(data['clean_text']).toarray().tolist()
data['FV_query'] = tfidf.transform(data['query']).toarray().tolist()
data['cosine_similarity'] = data.apply(lambda row: cosine_similarity([row['FV_query']],[row['FV_document']])[0][0], axis=1)
df = data[['query_cat', 'FV_query', 'FV_document', 'cosine_similarity', 'component_cat', 'system_cat', 'property_cat', 'relevance']].copy()
df.head()

,query_cat,FV_query,FV_document,cosine_similarity,component_cat,system_cat,property_cat,relevance
0,1,"[0.0, 0.0, 0.0, 0.0, 0.5645782010909166, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0.000000,20,8,3,0
1,1,"[0.0, 0.0, 0.0, 0.0, 0.5645782010909166, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.6802009394700788, 0.0, ...",0.384027,11,0,11,0
2,1,"[0.0, 0.0, 0.0, 0.0, 0.5645782010909166, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.5645782010909166, 0.0, ...",0.318749,51,0,14,0
3,1,"[0.0, 0.0, 0.0, 0.0, 0.5645782010909166, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0.000000,53,4,12,0
4,1,"[0.0, 0.0, 0.0, 0.0, 0.5645782010909166, 0.0, ...","[0.0, 0.0, 0.0, 0.7554534739534169, 0.0, 0.0, ...",0.000000,12,8,3,0


In [7]:
feature_columns = [
    'cosine_similarity',
    'component_cat',
    'system_cat',
    'property_cat'
    #'bm25',
    #'jaccard',
    #'term_overlap',
    #'semantic_similarity'
]

df['features'] = df[feature_columns].values.tolist()
df.head()

,query_cat,FV_query,FV_document,cosine_similarity,component_cat,system_cat,property_cat,relevance,features
0,1,"[0.0, 0.0, 0.0, 0.0, 0.5645782010909166, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0.000000,20,8,3,0,"[0.0, 20.0, 8.0, 3.0]"
1,1,"[0.0, 0.0, 0.0, 0.0, 0.5645782010909166, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.6802009394700788, 0.0, ...",0.384027,11,0,11,0,"[0.3840266227863685, 11.0, 0.0, 11.0]"
2,1,"[0.0, 0.0, 0.0, 0.0, 0.5645782010909166, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.5645782010909166, 0.0, ...",0.318749,51,0,14,0,"[0.3187485451470555, 51.0, 0.0, 14.0]"
3,1,"[0.0, 0.0, 0.0, 0.0, 0.5645782010909166, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0.000000,53,4,12,0,"[0.0, 53.0, 4.0, 12.0]"
4,1,"[0.0, 0.0, 0.0, 0.0, 0.5645782010909166, 0.0, ...","[0.0, 0.0, 0.0, 0.7554534739534169, 0.0, 0.0, ...",0.000000,12,8,3,0,"[0.0, 12.0, 8.0, 3.0]"


In [8]:
from scipy.sparse import csr_matrix
from sklearn.model_selection import GroupShuffleSplit

split1 = GroupShuffleSplit(n_splits=1,test_size=0.20,random_state=23)
train_idx, test_idx = next(split1.split(df, groups=df['query_cat']))

df_train = df.iloc[train_idx].copy()
df_test = df.iloc[test_idx].copy()

In [9]:
X_train = csr_matrix(df_train['features'].tolist())
y_train = df_train['relevance'].to_numpy()
qid_train = df_train['query_cat'].to_numpy()

X_test = csr_matrix(df_test['features'].tolist())
y_test = df_test['relevance'].to_numpy()
qid_test = df_test['query_cat'].to_numpy()

X_train.shape, X_test.shape, y_train.shape, y_test.shape, qid_train.shape, qid_test.shape

((134, 4), (67, 4), (134,), (67,), (134,), (67,))

In [10]:
from adarank import AdaRank
from metrics import NDCGScorer, APScorer

scorer = NDCGScorer(k=10)
#scorer = APScorer()
model = AdaRank(max_iter=100, estop=10, scorer=scorer).fit(X_train, y_train, qid_train)
pred = model.predict(X_test, qid_test)
print(scorer(y_test, pred, qid_test).mean())

0.9328502177967409


In [11]:
#'glucose in blood'
df_query = df_test[df_test['query_cat'] == 1].copy()
X_query = csr_matrix(df_query['features'].tolist())
scores = model.predict(X_query, [1] * len(df_query))
df_query['ranking_score'] = scores
df_query = df_query.sort_values(by='ranking_score', ascending=False)
df_query.head()

,query_cat,FV_query,FV_document,cosine_similarity,component_cat,system_cat,property_cat,relevance,features,ranking_score
23,1,"[0.0, 0.0, 0.0, 0.0, 0.5645782010909166, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.4296564610240364, 0.0, ...",0.761022,37,9,3,2,"[0.7610220518500801, 37.0, 9.0, 3.0]",0.412299
28,1,"[0.0, 0.0, 0.0, 0.0, 0.5645782010909166, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.88613768...",0.731400,38,8,11,1,"[0.7313999511518468, 38.0, 8.0, 11.0]",0.396250
12,1,"[0.0, 0.0, 0.0, 0.0, 0.5645782010909166, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.80493975...",0.664381,37,11,11,1,"[0.6643808390787261, 37.0, 11.0, 11.0]",0.359941
19,1,"[0.0, 0.0, 0.0, 0.0, 0.5645782010909166, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.80493975...",0.664381,37,6,11,1,"[0.6643808390787261, 37.0, 6.0, 11.0]",0.359941
18,1,"[0.0, 0.0, 0.0, 0.0, 0.5645782010909166, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.69105611...",0.570384,37,8,11,1,"[0.5703836100409061, 37.0, 8.0, 11.0]",0.309016


In [12]:
# Lista de documentos ordenados por relevancia dada una consulta
data.loc[df_query.index][:10]

,query,long_common_name,component,system,property,relevance,query_cat,component_cat,system_cat,property_cat,removed_punc,tokens,clean_tokens,lemma_words,clean_text,FV_document,FV_query,cosine_similarity
23,glucose in blood,"Glucose [Mass/volume] in Serum, Plasma or Blood",Glucose,Ser/Plas/Bld,MCnc,2,1,37,9,3,Glucose Massvolume in Serum Plasma or Blood,"[glucose, massvolume, in, serum, plasma, or, b...","[glucose, massvolume, serum, plasma, blood]","[glucose, massvolume, serum, plasma, blood]",glucose massvolume serum plasma blood,"[0.0, 0.0, 0.0, 0.0, 0.4296564610240364, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.5645782010909166, 0.0, ...",0.761022
28,glucose in blood,Glucose [Moles/volume] in Serum or Plasma --3 ...,Glucose^3H post 100 g glucose PO,Ser/Plas,SCnc,1,1,38,8,11,Glucose Molesvolume in Serum or Plasma 3 hours...,"[glucose, molesvolume, in, serum, or, plasma, ...","[glucose, molesvolume, serum, plasma, 3, hours...","[glucose, molesvolume, serum, plasma, 3, hour,...",glucose molesvolume serum plasma 3 hour post 1...,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.88613768...","[0.0, 0.0, 0.0, 0.0, 0.5645782010909166, 0.0, ...",0.731400
12,glucose in blood,Glucose [Moles/volume] in Urine,Glucose,Urine,SCnc,1,1,37,11,11,Glucose Molesvolume in Urine,"[glucose, molesvolume, in, urine]","[glucose, molesvolume, urine]","[glucose, molesvolume, urine]",glucose molesvolume urine,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.80493975...","[0.0, 0.0, 0.0, 0.0, 0.5645782010909166, 0.0, ...",0.664381
19,glucose in blood,Glucose [Moles/volume] in Pleural fluid,Glucose,Plr fld,SCnc,1,1,37,6,11,Glucose Molesvolume in Pleural fluid,"[glucose, molesvolume, in, pleural, fluid]","[glucose, molesvolume, pleural, fluid]","[glucose, molesvolume, pleural, fluid]",glucose molesvolume pleural fluid,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.80493975...","[0.0, 0.0, 0.0, 0.0, 0.5645782010909166, 0.0, ...",0.664381
18,glucose in blood,Glucose [Moles/volume] in Serum or Plasma,Glucose,Ser/Plas,SCnc,1,1,37,8,11,Glucose Molesvolume in Serum or Plasma,"[glucose, molesvolume, in, serum, or, plasma]","[glucose, molesvolume, serum, plasma]","[glucose, molesvolume, serum, plasma]",glucose molesvolume serum plasma,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.69105611...","[0.0, 0.0, 0.0, 0.0, 0.5645782010909166, 0.0, ...",0.570384
52,glucose in blood,Lymphocytes/100 leukocytes in Blood,Lymphocytes/100 leukocytes,Bld,NFr,0,1,46,0,8,Lymphocytes100 leukocytes in Blood,"[lymphocytes100, leukocytes, in, blood]","[lymphocytes100, leukocytes, blood]","[lymphocytes100, leukocyte, blood]",lymphocytes100 leukocyte blood,"[0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.5645782010909166, 0.0, ...",0.564578
55,glucose in blood,Fasting glucose [Mass or Moles/volume] in Seru...,Glucose^post CFst,Ser/Plas,MSCnc,0,1,39,8,6,Fasting glucose Mass or Molesvolume in Serum o...,"[fasting, glucose, mass, or, molesvolume, in, ...","[fasting, glucose, mass, molesvolume, serum, p...","[fasting, glucose, mass, molesvolume, serum, p...",fasting glucose mass molesvolume serum plasma,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.53360746...","[0.0, 0.0, 0.0, 0.0, 0.5645782010909166, 0.0, ...",0.440429
41,glucose in blood,Chloride [Moles/volume] in Blood,Chloride,Bld,SCnc,0,1,28,0,11,Chloride Molesvolume in Blood,"[chloride, molesvolume, in, blood]","[chloride, molesvolume, blood]","[chloride, molesvolume, blood]",chloride molesvolume blood,"[0.0, 0.0, 0.0, 0.0, 0.6802009394700788, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.5645782010909166, 0.0, ...",0.384027
1,glucose in blood,Bicarbonate [Moles/volume] in Blood,Bicarbonate,Bld,SCnc,0,1,11,0,11,Bicarbonate Molesvolume in Blood,"[bicarbonate, molesvolume, in, blood]","[bicarbonate, molesvolume, blood]","[bicarbonate, molesvolume, blood]",bicarbonate molesvolume blood,"[0.0, 0.0, 0.0, 0.0, 0.6802009394700788, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.5645782010909166, 0.0, ...",0.384027
29,glucose in blood,Calcium.ionized [Moles/volume] in Blood,Calcium.ionized,Bld

### Nueva consulta

In [13]:
new_query = ['platelets in blood']
df_new = pd.DataFrame([new_query], columns=['query'])
df_new['query_cat'] = encoders['query'].classes_.size
df_new['FV_query'] = tfidf.transform(df_new['query']).toarray().tolist()
df_new.head()

,query,query_cat,FV_query
0,platelets in blood,3,"[0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, ..."


In [14]:
df_new = pd.concat([df_new] * len(data['FV_document']), ignore_index=True)
df_new = pd.concat([df_new, data[['component_cat', 'system_cat', 'property_cat', 'FV_document']]], axis=1)
df_new.head()

,query,query_cat,FV_query,component_cat,system_cat,property_cat,FV_document
0,platelets in blood,3,"[0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, ...",20,8,3,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
1,platelets in blood,3,"[0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, ...",11,0,11,"[0.0, 0.0, 0.0, 0.0, 0.6802009394700788, 0.0, ..."
2,platelets in blood,3,"[0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, ...",51,0,14,"[0.0, 0.0, 0.0, 0.0, 0.5645782010909166, 0.0, ..."
3,platelets in blood,3,"[0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, ...",53,4,12,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
4,platelets in blood,3,"[0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, ...",12,8,3,"[0.0, 0.0, 0.0, 0.7554534739534169, 0.0, 0.0, ..."


In [15]:
df_new['cosine_similarity'] = df_new.apply(lambda row: cosine_similarity([row['FV_query']],[row['FV_document']])[0][0], axis=1)
df_new = df_new[['query_cat', 'FV_query', 'FV_document', 'cosine_similarity', 'component_cat', 'system_cat', 'property_cat']].copy()
df_new['features'] = df_new[feature_columns].values.tolist()
df_new.head()

,query_cat,FV_query,FV_document,cosine_similarity,component_cat,system_cat,property_cat,features
0,3,"[0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0.000000,20,8,3,"[0.0, 20.0, 8.0, 3.0]"
1,3,"[0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.6802009394700788, 0.0, ...",0.680201,11,0,11,"[0.6802009394700788, 11.0, 0.0, 11.0]"
2,3,"[0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.5645782010909166, 0.0, ...",0.564578,51,0,14,"[0.5645782010909166, 51.0, 0.0, 14.0]"
3,3,"[0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0.000000,53,4,12,"[0.0, 53.0, 4.0, 12.0]"
4,3,"[0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.7554534739534169, 0.0, 0.0, ...",0.000000,12,8,3,"[0.0, 12.0, 8.0, 3.0]"


In [16]:
X_new = csr_matrix(df_new['features'].tolist())
scores = model.predict(X_new, [3] * len(df_new))
df_new['ranking_score'] = scores
df_new = df_new.sort_values(by='ranking_score', ascending=False)
df_new.head()

,query_cat,FV_query,FV_document,cosine_similarity,component_cat,system_cat,property_cat,features,ranking_score
52,3,"[0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, ...",1.000000,46,0,8,"[1.0, 46.0, 0.0, 8.0]",0.541770
165,3,"[0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, ...",1.000000,46,0,8,"[1.0, 46.0, 0.0, 8.0]",0.541770
106,3,"[0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, ...",1.000000,46,0,8,"[1.0, 46.0, 0.0, 8.0]",0.541770
80,3,"[0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.6802009394700788, 0.0, ...",0.680201,23,0,11,"[0.6802009394700788, 23.0, 0.0, 11.0]",0.368512
152,3,"[0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.6802009394700788, 0.0, ...",0.680201,11,0,11,"[0.6802009394700788, 11.0, 0.0, 11.0]",0.368512


In [17]:
# Lista de documentos ordenados por relevancia dada una consulta que el modelo no vio en el entrenamiento
data.loc[df_new.index][:10]

,query,long_common_name,component,system,property,relevance,query_cat,component_cat,system_cat,property_cat,removed_punc,tokens,clean_tokens,lemma_words,clean_text,FV_document,FV_query,cosine_similarity
52,glucose in blood,Lymphocytes/100 leukocytes in Blood,Lymphocytes/100 leukocytes,Bld,NFr,0,1,46,0,8,Lymphocytes100 leukocytes in Blood,"[lymphocytes100, leukocytes, in, blood]","[lymphocytes100, leukocytes, blood]","[lymphocytes100, leukocyte, blood]",lymphocytes100 leukocyte blood,"[0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.5645782010909166, 0.0, ...",0.564578
165,white blood cells count,Lymphocytes/100 leukocytes in Blood,Lymphocytes/100 leukocytes,Bld,NFr,2,2,46,0,8,Lymphocytes100 leukocytes in Blood,"[lymphocytes100, leukocytes, in, blood]","[lymphocytes100, leukocytes, blood]","[lymphocytes100, leukocyte, blood]",lymphocytes100 leukocyte blood,"[0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, ...",1.000000
106,bilirrubin in plasma,Lymphocytes/100 leukocytes in Blood,Lymphocytes/100 leukocytes,Bld,NFr,0,0,46,0,8,Lymphocytes100 leukocytes in Blood,"[lymphocytes100, leukocytes, in, blood]","[lymphocytes100, leukocytes, blood]","[lymphocytes100, leukocyte, blood]",lymphocytes100 leukocyte blood,"[0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0.000000
80,bilirrubin in plasma,Calcium.ionized [Moles/volume] in Blood,Calcium.ionized,Bld,SCnc,0,0,23,0,11,Calciumionized Molesvolume in Blood,"[calciumionized, molesvolume, in, blood]","[calciumionized, molesvolume, blood]","[calciumionized, molesvolume, blood]",calciumionized molesvolume blood,"[0.0, 0.0, 0.0, 0.0, 0.6802009394700788, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0.000000
152,white blood cells count,Bicarbonate [Moles/volume] in Blood,Bicarbonate,Bld,SCnc,0,2,11,0,11,Bicarbonate Molesvolume in Blood,"[bicarbonate, molesvolume, in, blood]","[bicarbonate, molesvolume, blood]","[bicarbonate, molesvolume, blood]",bicarbonate molesvolume blood,"[0.0, 0.0, 0.0, 0.0, 0.6802009394700788, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, ...",0.680201
1,glucose in blood,Bicarbonate [Moles/volume] in Blood,Bicarbonate,Bld,SCnc,0,1,11,0,11,Bicarbonate Molesvolume in Blood,"[bicarbonate, molesvolume, in, blood]","[bicarbonate, molesvolume, blood]","[bicarbonate, molesvolume, blood]",bicarbonate molesvolume blood,"[0.0, 0.0, 0.0, 0.0, 0.6802009394700788, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.5645782010909166, 0.0, ...",0.384027
41,glucose in blood,Chloride [Moles/volume] in Blood,Chloride,Bld,SCnc,0,1,28,0,11,Chloride Molesvolume in Blood,"[chloride, molesvolume, in, blood]","[chloride, molesvolume, blood]","[chloride, molesvolume, blood]",chloride molesvolume blood,"[0.0, 0.0, 0.0, 0.0, 0.6802009394700788, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.5645782010909166, 0.0, ...",0.384027
167,white blood cells count,"Carbon dioxide, total [Moles/volume] in Blood",Carbon dioxide,Bld,SCnc,0,2,25,0,11,Carbon dioxide total Molesvolume in Blood,"[carbon, dioxide, total, molesvolume, in, blood]","[carbon, dioxide, total, molesvolume, blood]","[carbon, dioxide, total, molesvolume, blood]",carbon dioxide total molesvolume blood,"[0.0, 0.0, 0.0, 0.0, 0.6802009394700788, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, ...",0.680201
158,white blood cells count,Chloride [Moles/volume] in Blood,Chloride,Bld,SCnc,0,2,28,0,11,Chloride Molesvolume in Blood,"[chloride, molesvolume, in, blood]","[chloride, molesvolume, blood]","[chloride, molesvolume, blood]",chloride molesvolume blood,"[0.0, 0.0, 0.0, 0.0, 0.6802009394700788, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, ...",0.680201
29,glucose in blood,Calcium.ionized [Moles/volume] in Blood,Calcium.ionized,Bld,SCnc,0,1,23,0,11,Calciumionized Molesvolume in Blood,"[calciumionized, molesvolume, in, blood]","[calciumionized, molesvolume, blood]","[calciumionized, molesvolume, blood]",

### Dataset con más terminos

### Dataset con más consultas